# Exportação para Relacional: Delta Lake ➔ Azure SQL Server
Este notebook executa:
- Leitura dos parâmetros de conexão do banco via `.env`.
- Limpeza opcional de metadados internos de ingestão (`source_file`, `ingestion_time`).
- Carga no Azure SQL Server via conector JDBC.
- Leitura de validação pós-carga para assegurar o número de registros gravados.

In [0]:
import os
from dotenv import load_dotenv

# ==============================================================================
# 1. Carregamento de Configurações e Conexão JDBC
# ==============================================================================
load_dotenv(".env")

jdbc_hostname = os.getenv("SQL_HOST")
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

assert jdbc_hostname and jdbc_database and jdbc_username and jdbc_password, (
    "Credenciais do SQL Server não encontradas no arquivo .env."
)

jdbc_port = 1433
jdbc_url = (
    f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};"
    f"database={jdbc_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

# Tabela de destino no padrão squadX.nome_tabela
target_table = "squad1.ecommerce_enderecos_luiz_riuler"

In [0]:
# ==============================================================================
# 2. Leitura da Tabela Delta e Preparação de Dados
# ==============================================================================
df_enderecos = spark.table("raw_ecommerce_enderecos")

# Removendo colunas técnicas de auditoria antes da exportação
df_enderecos_export = df_enderecos.drop("source_file", "ingestion_time")

print(f"Total de registros a serem enviados: {df_enderecos_export.count()}")

In [0]:
# ==============================================================================
# 3. Escrita no Banco Relacional via Conector SQL Server
# ==============================================================================
(
    df_enderecos_export.write
    .format("sqlserver")
    .option("host", jdbc_hostname)
    .option("port", jdbc_port)
    .option("database", jdbc_database)
    .option("dbtable", target_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("encrypt", "true")
    .option("trustServerCertificate", "false")
    .mode("overwrite")
    .save()
)

print(f"✔ Dados gravados com sucesso na tabela '{target_table}'!")

In [0]:
# ==============================================================================
# 4. Validação Pós-Carga
# ==============================================================================
df_validacao = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", target_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

print(f"Total de registros validados no SQL Server: {df_validacao.count()}")
display(df_validacao.limit(10))